# Fraud model v3 - exploration + scoring

Quick notebook to load the trained model and score a batch of transactions.
(Aisha, 2026-06-30 - don't judge the mess, it works :))


In [ ]:
import pandas as pd
import joblib
import math
import os

# SMELL: model loaded at import time - every tool that imports this pays the cost,
# and there is no way to control WHEN loading happens or handle a failure.
model_bundle = joblib.load("models/fraud_xgb_v3.joblib")
model = model_bundle["pipeline"]
MODEL_VERSION = model_bundle["version"]
print("loaded", MODEL_VERSION)


## Load the data

In [ ]:
# SMELL: hardcoded path from my laptop. Works "on my machine" only.
# df = pd.read_csv("/Users/aisha/Desktop/fraud-project/data/transactions_sample.csv")
df = pd.read_csv("data/transactions_sample.csv")
df.head()


## Feature engineering

amount needs to be log-scaled because of the crazy outliers, channel/mcc are categorical,
and night-time transactions look riskier from the histogram I made last week.

In [ ]:
# SMELL: threshold and feature list are just... here. Nobody knows this is the
# business decision boundary; it will get typed differently in three other files.
BLOCK_THRESHOLD = float(os.environ.get("THRESHOLD", 0.85))
FEATURE_COLS = ["amount_log", "channel", "mcc", "hour_of_day", "is_night"]

df["amount_log"] = df["amount_sar"].apply(lambda x: math.log1p(x))
df["mcc"] = df["merchant_category"].str.strip().str.upper().str.replace(" ", "_")
df["hour_of_day"] = pd.to_datetime(df["timestamp"]).dt.hour
df["is_night"] = (df["hour_of_day"] < 6).astype(int)
df[FEATURE_COLS].head()


## Score everything

Loop is a bit slow but 5k rows is nothing. Wrapped it in try/except because a few rows were causing weird errors and I didn't want the whole run to die.

In [ ]:
# SMELL: bare except swallows real bugs and silently returns a "safe" default -
# in production this means a fraud check quietly stops checking anything.
results = []
for _, row in df.iterrows():
    try:
        features = row[FEATURE_COLS].to_frame().T
        proba = model.predict_proba(features)[0, 1]
    except Exception:
        proba = 0.0  # "probably fine"
    decision = "block" if proba >= BLOCK_THRESHOLD else (
        "review" if proba >= BLOCK_THRESHOLD - 0.15 else "allow")
    results.append({"transaction_id": row["transaction_id"], "score": proba,
                    "decision": decision})

scored = pd.DataFrame(results)
scored["decision"].value_counts()


## Wait, let me double check the log transform

Saw a blog post recommend log10 instead of log1p for skewed money amounts, let me try it here real quick and compare.

In [ ]:
# SMELL (execution-order trap 1/3): this OVERWRITES amount_log in place using a
# different formula than training used. Re-run cell 4 after this and every score
# silently drifts - this is training/serving skew, self-inflicted, in one notebook.
df["amount_log"] = df["amount_sar"].apply(lambda x: math.log10(x + 1))
df[["amount_sar", "amount_log"]].head()


## Actually, let's also try dropping tiny transactions - noise


In [ ]:
# SMELL (execution-order trap 2/3): mutates the shared df/threshold globals that
# earlier AND later cells both read. "Run All" gives different numbers than
# clicking cells one at a time in the order I actually used while building this.
df = df[df["amount_sar"] >= 10].reset_index(drop=True)
BLOCK_THRESHOLD = 0.80  # felt right after looking at the review-band counts above
print(len(df), "rows left, threshold now", BLOCK_THRESHOLD)


## Re-score with the filtered data + new threshold

In [ ]:
# SMELL (execution-order trap 3/3): this cell LOOKS identical to cell 4 but produces
# different numbers depending on which of cells 5/6 already ran, and in what order.
# There is no way to know from reading this cell alone. That is the whole problem.
results = []
for _, row in df.iterrows():
    try:
        features = row[FEATURE_COLS].to_frame().T
        proba = model.predict_proba(features)[0, 1]
    except Exception:
        proba = 0.0
    decision = "block" if proba >= BLOCK_THRESHOLD else (
        "review" if proba >= BLOCK_THRESHOLD - 0.15 else "allow")
    results.append({"transaction_id": row["transaction_id"], "score": proba,
                    "decision": decision})

scored = pd.DataFrame(results)
scored.to_csv("scored.csv", index=False)
scored["decision"].value_counts()


## TODO before demo
- [ ] ask data team why some rows blow up in predict_proba
- [ ] this works locally but Karim says it crashed on his machine, no idea why
- [ ] maybe move this to a .py file at some point?
